# 02 | Coorte final e dataset supervisionado

Fluxo: auditar → candidata → publicar → artefato supervisionado.

Contrato: `configs/final_cohort_v1.yaml`. Partição: `configs/supervised.yaml` (`temporal`). Blocos 70/15/15 em 2019 e 2023–2025; 2020–2022 em `test_pandemic`; 2026 em `exploracao`. Testes fechados até o G9.

In [3]:
import json, shutil, sys, os, gc, subprocess, hashlib
from pathlib import Path
import psutil

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs" / "supervised.yaml").exists())
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
for m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[m]

import pandas as pd
from IPython.display import display
from src.dados import gerar_coorte
from src.dados import gerar_artefato_supervisionado

def mem_gb():
    return psutil.Process().memory_info().rss / 1e9

def existe(p):
    return p.exists() and p.stat().st_size > 0

def hash_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

CAND = ROOT / "outputs/coorte_candidata"
PUB  = ROOT / "data/analytical/coorte_final.parquet"
ART  = ROOT / "data/analytical/dataset_supervisionado.parquet"
MAN  = ROOT / "data/analytical/manifesto_supervisionado.json"

print(f"RAM inicial: {mem_gb():.1f} GB")
print(f"candidata existe?         {existe(CAND / 'coorte_final.parquet')}")
print(f"coorte publicada existe?  {existe(PUB)}")
print(f"artefato existe?          {existe(ART)}")
print(f"manifesto existe?         {existe(MAN)}")

GERAR_CANDIDATA  = True
PUBLICAR         = True
GERAR_ARTEFATO   = not (existe(ART) and existe(MAN))
USAR_SUBPROCESSO = True

REF_24_10 = {"train_ref": 556_462, "val_ref": 119_243,
             "test_ref": 119_245, "test_pandemic": 2_915_829}

if not GERAR_ARTEFATO and existe(ART) and existe(MAN):
    _man = json.loads(MAN.read_text(encoding="utf-8"))
    _esp = _man.get("dataset_sha256", "")
    _atu = hash_file(ART)
    _ok  = (_atu == _esp)
    print(f"artefato: cohort_kind = {_man.get('cohort_kind')}")
    print(f"artefato: features = {len(_man.get('feature_columns', []))}")
    print(f"artefato: hash {'OK' if _ok else 'DIVERGE'} (esperado {_esp[:12]}, obtido {_atu[:12]})")
    del _man, _esp, _atu, _ok

print("Raiz:", ROOT)

RAM inicial: 0.2 GB
candidata existe?         False
coorte publicada existe?  False
artefato existe?          False
manifesto existe?         False
Raiz: c:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador


In [4]:
if GERAR_CANDIDATA:
    gerar_coorte("audit", sobrescrever=True)
else:
    print("Auditoria pulada (GERAR_CANDIDATA=False).")

_resumo = CAND / "resumo_regras.csv"
if _resumo.exists():
    display(pd.read_csv(_resumo))
else:
    print("resumo_regras.csv ainda não existe.")
del _resumo
gc.collect()
print(f"RAM após auditoria: {mem_gb():.1f} GB")

Fonte: C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\data\processed\dados-v2\srag.parquet
Contrato: C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\configs\final_cohort_v1.yaml
Colunas da auditoria: ['hospital', 'dt_interna', 'dt_sin_pri', 'evolucao']
Linhas carregadas: 4666910

Auditoria concluída.
Regras aprovadas para exclusão: {'hospital_nao_confirmado': 232253, 'dt_interna_invalida': 304245, 'evolucao_fora_alvo_principal': 530370, 'dt_interna_antes_periodo': 48, 'dt_interna_apos_periodo': 76, 'intervalo_sintomas_negativo': 24361, 'intervalo_sintomas_maior_limite': 11708}
Regras pendentes de decisão: {}
Diretório dos resultados: C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\outputs\coorte_candidata
Nenhuma exclusão foi aplicada.
Nenhuma candidata foi gerada.
Nenhuma coorte foi publicada.


,regra,linhas_sinalizadas,total_linhas,proporcao_fonte
0,hospital_nao_confirmado,232253,4666910,0.049766
1,dt_interna_invalida,304245,4666910,0.065192
2,evolucao_fora_alvo_principal,530370,4666910,0.113645
3,dt_interna_antes_periodo,48,4666910,0.000010
4,dt_interna_apos_periodo,76,4666910,0.000016
5,intervalo_sintomas_negativo,24361,4666910,0.005220
6,intervalo_sintomas_maior_limite,11708,4666910,0.002509


RAM após auditoria: 0.8 GB


In [5]:
if GERAR_CANDIDATA:
    gerar_coorte("candidate", sobrescrever=True)

_val_path   = CAND / "validacao_publicacao.json"
_water_path = CAND / "coorte_waterfall.csv"

if _val_path.exists() and _water_path.exists():
    val = json.loads(_val_path.read_text(encoding="utf-8"))
    rec = val["population_reconciliation"]
    print(rec)
    assert rec["closed"], "waterfall não fecha: não publique"
    display(pd.read_csv(_water_path))
else:
    val = rec = None
    print("validacao_publicacao.json ou coorte_waterfall.csv ainda não existe.")

del _val_path, _water_path
gc.collect()
print(f"RAM após candidata: {mem_gb():.1f} GB")

Fonte: C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\data\processed\dados-v2\srag.parquet
Contrato: C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\configs\final_cohort_v1.yaml
Row groups: 28
Linhas da fonte: 4666910
Row group 1/28: fonte=200000, candidata=155294
Row group 2/28: fonte=20941, candidata=16240
Row group 3/28: fonte=200000, candidata=171843
Row group 4/28: fonte=136420, candidata=117146
Row group 5/28: fonte=200000, candidata=168924
Row group 6/28: fonte=67986, candidata=57355
Row group 7/28: fonte=200000, candidata=169011
Row group 8/28: fonte=79453, candidata=67269
Row group 9/28: fonte=200000, candidata=163212
Row group 10/28: fonte=200000, candidata=163782
Row group 11/28: fonte=160577, candidata=131240
Row group 12/28: fonte=200000, candidata=165982
Row group 13/28: fonte=200000, candidata=165926
Row group 14/28: fonte=200000, candidata=166236
Row group 15/28: fonte=200000, candidata=165986
Row group 16/28: fonte=200000, candidata=16602

,ordem,regra,excluidas_na_etapa,restantes_apos_etapa
0,1,hospital_nao_confirmado,232253,4434657
1,2,dt_interna_invalida,72016,4362641
2,3,evolucao_fora_alvo_principal,447436,3915205
3,4,dt_interna_antes_periodo,29,3915176
4,5,dt_interna_apos_periodo,34,3915142
5,6,intervalo_sintomas_negativo,20918,3894224
6,7,intervalo_sintomas_maior_limite,9496,3884728


RAM após candidata: 0.9 GB


In [6]:
if PUBLICAR:
    _src = CAND / "coorte_final.parquet"
    if not _src.exists():
        print(f"Nada para publicar: {_src} não existe (rode GERAR_CANDIDATA=True primeiro).")
    else:
        PUB.parent.mkdir(parents=True, exist_ok=True)
        if PUB.exists():
            shutil.copy2(PUB, PUB.with_suffix(".parquet.bak"))
        shutil.copy2(_src, PUB)

        _mp = PUB.with_name("coorte_final.manifest.json")
        shutil.copy2(CAND / "manifesto_coorte.json", _mp)

        _m = json.loads(_mp.read_text(encoding="utf-8"))
        _m["publication_performed"] = True
        _m["legacy_parity_reviewed"] = True
        _m["provisional_comparison_reviewed"] = True
        _mp.write_text(json.dumps(_m, indent=2, ensure_ascii=False), encoding="utf-8")
        del _mp, _m
    del _src

if PUB.exists():
    print(f"Coorte publicada: {PUB} ({PUB.stat().st_size/1e6:.1f} MB)")
else:
    print(f"Coorte ainda não publicada em {PUB}")

for _n in ("val", "rec"):
    if _n in globals():
        del globals()[_n]
gc.collect()
try:
    import pyarrow as pa
    pa.default_memory_pool().release_unused()
except Exception:
    pass
print(f"RAM após publicar: {mem_gb():.1f} GB")

Coorte publicada: c:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\data\analytical\coorte_final.parquet (255.0 MB)
RAM após publicar: 0.5 GB


In [7]:
if GERAR_ARTEFATO:
    if USAR_SUBPROCESSO:
        _code = (
            "from pathlib import Path; "
            "from src.dados import gerar_artefato_supervisionado; "
            "gerar_artefato_supervisionado(Path('.'), cohort_kind='final', "
            "exportar_particao=True, overwrite=True); "
            "print('OK')"
        )
        print("Rodando em subprocesso (memória liberada ao terminar o processo)...")
        _env = {**os.environ, "PYTHONIOENCODING": "utf-8"}
        _proc = subprocess.Popen(
            [sys.executable, "-c", _code],
            cwd=str(ROOT),
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding="utf-8", errors="replace",
            bufsize=1, env=_env,
        )
        for _line in _proc.stdout:
            print(_line, end="", flush=True)
        _proc.stdout.close()
        _proc.wait()
        _rc = _proc.returncode
        del _proc, _code, _env
        if _rc != 0:
            raise RuntimeError(f"gerar_artefato_supervisionado falhou (código {_rc})")
    else:
        print("Rodando no kernel (vai consumir RAM; reinicie o kernel antes, se possível)...")
        gerar_artefato_supervisionado(ROOT, cohort_kind="final", exportar_particao=True, overwrite=True)

contagens = None
if MAN.exists():
    _man = json.loads(MAN.read_text(encoding="utf-8"))
    contagens = _man.get("partition_counts")
    del _man

if contagens is not None:
    tab = pd.DataFrame({"linhas": pd.Series(contagens), "ref_24_10": pd.Series(REF_24_10)})
    tab["igual_ref"] = tab["linhas"] == tab["ref_24_10"]
    display(tab)
else:
    print("Sem manifesto; contagens indisponíveis.")

gc.collect()
print(f"RAM após o artefato: {mem_gb():.1f} GB")

Rodando em subprocesso (memória liberada ao terminar o processo)...
OK


,linhas,ref_24_10,igual_ref
test_pandemic,2915829,2915829,True
test_ref,119245,119245,True
train_ref,556462,556462,True
val_ref,119243,119243,True


RAM após o artefato: 0.5 GB


In [8]:
if contagens is not None:
    comp = {k: v for k, v in contagens.items() if k in ("train_ref", "val_ref", "test_ref")}
    pct = pd.Series(comp) / sum(comp.values())
    esperado = pd.Series({"train_ref": .70, "val_ref": .15, "test_ref": .15})
    display(pd.DataFrame({"pct": pct.round(4), "esperado": esperado}))
    assert ((pct - esperado).abs() < 0.005).all(), "razão fora de 70/15/15 (tolerância 0,5 p.p.)"
    print("OK: 70/15/15. Testes seguem fechados até o G9.")
else:
    print("Sem contagens; rode a célula do artefato primeiro.")

,pct,esperado
test_ref,0.15,0.15
train_ref,0.70,0.70
val_ref,0.15,0.15


OK: 70/15/15. Testes seguem fechados até o G9.
